# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all record sets and their fields, referencing each by its `@id`:

In [ ]:
# List all record sets and their fields by @id
record_sets = metadata.record_sets
if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"Record Set '@id': {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        print("  Fields (by @id):")
        for field in rs.get('fields', []):
            print(f"    - {field['@id']} | Name: {field.get('name', '')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

To ensure we reference entities by their `@id`, we select one or more record set `@id`s for loading. If only one record set is present, we use that.

In [ ]:
# Extract data from each record set (referenced by @id)
import pprint

record_sets = metadata.record_sets

record_set_ids = []
if record_sets:
    for rs in record_sets:
        record_set_ids.append(rs['@id'])
else:
    # If no explicit record sets, try to enumerate files or use the Dataset directly
    print("No record sets found to load data from.")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}:")
    pprint.pprint(list(df.columns))
    print(df.head(3))

# For reproducibility in the EDA below, select the first found record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. Select fields (by `@id`) for operations below.

Modify field `@id` variables below to correspond to actual numerical and categorical fields in your dataset.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Running EDA for record set: {main_record_set_id}")
    # List all columns
    print("Columns available:")
    print(df.columns.tolist())
    
    # Example: Try the following likely numeric field @id. Update as needed!
    numeric_field = None
    group_field = None
    for col in df.columns:
        # Pick the first numeric-looking column
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    
    # Pick first string/categorical field, other than the numeric one
    for col in df.columns:
        if col != numeric_field and pd.api.types.is_string_dtype(df[col]):
            group_field = col
            break
    
    if numeric_field:
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records with {numeric_field} > mean ({threshold:.2f}):")
        print(filtered_df.head(3))
        
        filtered_df = filtered_df.copy()  # avoid pandas SettingWithCopyWarning
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(3))
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head(3))
    else:
        print("No numeric field found for EDA.")
else:
    print("No data found to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and main_record_set_id in dataframes and numeric_field:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,6))
        df.boxplot(column=numeric_field, by=group_field, rot=90)
        plt.title(f'{numeric_field} by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric data available to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- We loaded metadata and identified available record sets and fields by their `@id` as defined in the Croissant schema.
- Data from selected record sets was read into DataFrames, and initial data inspection was performed.
- Basic exploratory data analysis and visualizations were executed: filtering, normalization, and group-wise statistics by field `@id`s.
- For advanced analysis, modify the field IDs in the notebook as needed and extend the EDA section.

Refer to the [FAIR^2 Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for details on available record sets and field `@id`s.